In [ ]:
import sys
import os


RESULTS_PATH = 'results.csv'
OPTUNA_DATABASE = 'study_results.db'
BASE_CONFIGURATION_NAME = 'configuration'


In [ ]:
from automl.loggers.result_logger import ResultLogger
import optuna
import optuna.visualization as vis
from automl.utils.optuna_utils import load_study_from_database
import matplotlib.pyplot as plt


# Load the experiment

In [ ]:
#base_experiment_path = "C:\\Experiments\\rl-zoo-CartPole-dqn-2\\HPOptimizationExperiments\\3\\experiments"
#experiment_relative_path = 'original'

#base_experiment_path = "C:\\ricardo-goncalo-thesis-project\\project\\examples\\hp_optimization\\data\\hp_lodaded_exps"
#experiment_relative_path = "exp_27"

#base_experiment_path = "C:\\Experiments\\rl-zoo-CartPole-ppo-multi_thread\\HpExperiments\\1"
#experiment_relative_path = "experiment"

#base_experiment_path = "C:\\Experiments\\hyperband_ppo_cartpole\\HpExperiments"
#experiment_relative_path = "8"

#base_experiment_path = "C:\\Experiments\\rl-zoo-CartPole-dqn-2\\HPOptimizationExperiments\\3\\experiments"
#experiment_relative_path = "original_adapted"

base_experiment_path = "C:\\Experiments\\hyperband_ppo_cartpole\\HpExperiments_v3"
experiment_relative_path = "1"

experiment_path = f'{base_experiment_path}\\{experiment_relative_path}'



In [ ]:
if not os.path.exists(experiment_path):
    raise Exception("DOES NOT EXIST")

In [ ]:
from automl.core.advanced_input_management import gen_component_from
from automl.hp_opt.hp_optimization_pipeline import HyperparameterOptimizationPipeline


hyperparameter_optimization_pipeline : HyperparameterOptimizationPipeline = gen_component_from(experiment_path)
hyperparameter_optimization_pipeline.change_logger_level("NONE") # we don't want any logging done while we're looking into it


In [ ]:
from automl.hp_opt.hp_opt_strategies.hp_optimization_loader_detached import HyperparameterOptimizationPipelineLoaderDetached

USE_MULTIPLE = isinstance(hyperparameter_optimization_pipeline, HyperparameterOptimizationPipelineLoaderDetached)

# Evaluation of HyperparameterOptimizationPipeline

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_hp_opt_results_logger

hyperparameter_optimization_results = hyperparameter_optimization_pipeline.get_decoupled_results_logger()

print(f"Hyperparameter_optimization_results in path: {hyperparameter_optimization_results.get_artifact_directory()}")

In [ ]:
if not USE_MULTIPLE:
    df = hyperparameter_optimization_results.get_dataframe()
    df["component_index"] = 0

In [ ]:
experiments_in_results = set(hyperparameter_optimization_results.get_dataframe()["experiment"].values)
component_indexes_in_results = set(hyperparameter_optimization_results.get_dataframe()["component_index"].values)

print(f"Experiments in results:\n{experiments_in_results}")

print(f"Component indexes per experiment:\n{component_indexes_in_results}")


In [ ]:
colors_available_for_component_indexes = ["red", "blue", "green"]
colors_for_component_indexes = {}

color_iter = iter(colors_available_for_component_indexes)
for component_index in component_indexes_in_results:
    colors_for_component_indexes[component_index] = next(color_iter)

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_hp_opt_optuna_study


optuna_study : optuna.Study = hyperparameter_optimization_pipeline.get_study()


In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_results_of_configurations_components                          



results_of_configurations : dict[str, dict[str, ResultLogger]]= get_results_of_configurations_components(experiment_path, use_multiple=USE_MULTIPLE)

# Smaller study

## Configuration study in optuna

In [ ]:
#trials_to_study = [0]


In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_trials_with_decreasing_intermediates

FILTER = "completed"

match FILTER:
    case "decreasing_inter":

        trials_to_study = get_trials_with_decreasing_intermediates(optuna_study)
        print(f"There are {len(trials_to_study)} trials with decreasing intermidiates")

    case "pruned":
        trials_to_study = [trial for trial in optuna_study.get_trials() if trial.state == optuna.trial.TrialState.PRUNED]

    case "completed":
        trials_to_study = [trial for trial in optuna_study.get_trials() if trial.state == optuna.trial.TrialState.COMPLETE]


    case _:
        trials_to_study = optuna_study.get_trials()


In [ ]:
trials_to_study.sort(key=lambda trial: trial.value) 

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import get_trials_with_decreasing_intermediates, print_intermidiate_values

print_intermidiate_values(trials_to_study)

In [ ]:
from automl.hp_opt.hp_eval_results.hp_eval_results import plot_scattered_values_for_all_params

if False:

    try:
        plot_scattered_values_for_all_params(optuna_study, trials_to_study)
    
    except Exception as e:
        print(f"Could not plot scattered values because of error: {e}")

## Results study

In [ ]:
configurations_to_study = [(f'configuration_{trial.number}', trial) for trial in trials_to_study]

In [ ]:

from automl.hp_opt.hp_eval_results.hp_eval_results import study_of_components_for_configuration, study_of_configuration

n_configurations_to_print = 20
USE_BEST = True


if n_configurations_to_print > 0:
    configurations_to_print = configurations_to_study[(len(configurations_to_study) - n_configurations_to_print):] if USE_BEST else configurations_to_study[:n_configurations_to_print] 

elif n_configurations_to_print == 0:
    configurations_to_print = []

else:
    configurations_to_print = configurations_to_study

try:

    for (configuration_name, trial) in configurations_to_print:
    
        results_loggers : dict = results_of_configurations[configuration_name]
    
        study_of_components_for_configuration(configuration_name, results_loggers, colors_for_component_indexes=colors_for_component_indexes)
        
        hyperparameter_optimization_results.plot_overlaid_bar(x_axis="step", y_axis='result', fixed_value_tuple=("experiment", trial.number), 
                                                              colors_for_value=("component_index", colors_for_component_indexes), title=f"{configuration_name}") 

        #for component_name in results_loggers.keys():
#
        #    study_of_configuration(f"{configuration_name}_{component_name}",
        #                           results_loggers[component_name],
        #                           )



except KeyError as e:
    print(f"KeyError: {e}\nAvailable keys are {results_of_configurations.keys()}")
    raise e

### Study for each agent

In [ ]:
agents_in_study = []
# agents_in_study = ["agent_1", "agent_2"]

In [ ]:
agents_to_study : dict[str, ResultLogger]= {}

for (configuration_name, trial) in configurations_to_study:
    
    results_logger = results_of_configurations[configuration_name]
    
    for agent_name in agents_in_study:
      
        agent_results_logger = ResultLogger(input={
                                            "logger_directory" : f"{results_logger.lg.logDir}\\{agent_name}",
                                            "filename" : RESULTS_PATH,
                                            "create_new_directory" : False
                                          })

        agents_to_study[f"{configuration_name}_{agent_name}"] = agent_results_logger
        
        agent_results_logger.proccess_input()


In [ ]:
for agent_name, agent_results_logger in agents_to_study.items():
    
    study_of_configuration(agent_name, agent_results_logger)